**Zadanie 1**


1. Korzystając z biblioteki pygrok, parse, re lub innej dowolnej, przenieś plik z logami do obiektu DataFrame, a treść logu zapisz do kolumny msg.

2. Bazując na ramce danych z zadania 1, sprawdź, w ilu rekordach występują wartości puste (None).

3. Przygotuj zestawienie liczby logów wygenerowanych przez poszczególne procesy (np. sshd, ftpd, su).

4. Na bazie przygotowanego wcześniej DataFrame, przekonwertuj kolumnę z datą i czasem ze zwykłego tekstu na typ datetime (w pliku nie podano roku, wstaw rok 2025).

5. Wykorzystując dowolną bibliotekę (pygrok, parse, re), wyciągnij z kolumny msg informacje o adresie IP i nazwie użytkownika dla komunikatów, które zawierają "Failed password" oraz "Invalid user".

6. Korzystając z metod filtracji i grupowania, zastosuj je do zidentyfikowania i wyświetlenia zestawienia top 5 adresów IP generujących najwięcej błędów uwierzytelniania oraz top 5 najczęściej atakowanych nazw użytkowników (np. root, admin). Wyniki wyeksportuj do pliku JSON.

In [17]:
import json
import re
import pandas as pd

# Wczytanie pliku z logami
with open("OpenSSH_2k.log", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

# 1: Parsowanie do DataFrame z kolumną 'msg'
pattern = r'^(?P<Month>\w+)\s+(?P<Day>\d+)\s+(?P<Time>\d{2}:\d{2}:\d{2})\s+(?P<Host>\S+)\s+(?P<Process>[\w\-]+)(?:\[\d+\])?:\s+(?P<msg>.*)$'
df = pd.DataFrame([re.match(pattern, line).groupdict() for line in lines if re.match(pattern, line)])
print("1.")
print(df.head())
print('\n')

# 2: Liczba rekordów z wartościami pustymi (None/NaN)
null_count = df.isna().any(axis=1).sum()
print("2.")
print(f"Liczba wierszy z brakującymi danymi: {null_count}")
print('\n')

# 3: Liczba logów wygenerowanych przez poszczególne procesy
process_counts = df['Process'].value_counts()
print("3.")
print(process_counts)
print('\n')

# 4: Konwersja na typ datetime z rokiem 2025
df['Datetime'] = pd.to_datetime('2025 ' + df['Month'] + ' ' + df['Day'] + ' ' + df['Time'], format='%Y %b %d %H:%M:%S')
print("4.")
print(df[['Datetime', 'Process', 'msg']].head())
print('\n')

# 5: Wyciągnięcie IP i użytkownika dla nieudanych logowań
auth_df = df[df['msg'].str.contains("Failed password|Invalid user", case=False, na=False)].copy()
auth_df['IP'] = auth_df['msg'].str.extract(r'from\s+(\d{1,3}(?:\.\d{1,3}){3})')
auth_df['User'] = auth_df['msg'].str.extract(r'(?:for\s+invalid\s+user\s+|for\s+|user\s+)(\S+)')
print("5.")
print(auth_df[['User', 'IP', 'msg']].head())
print('\n')

# 6: Top 5 IP, Top 5 użytkowników oraz eksport do pliku JSON
top_5_ips_df = (
    auth_df['IP']
    .value_counts()
    .head(5)
    .reset_index()
)
top_5_ips_df.columns = ['IP', 'Count']

top_5_users_df = (
    auth_df['User']
    .value_counts()
    .head(5)
    .reset_index()
)
top_5_users_df.columns = ['User', 'Count']

results = {
    "top_5_ips": dict(zip(top_5_ips_df['IP'], top_5_ips_df['Count'])),
    "top_5_users": dict(zip(top_5_users_df['User'], top_5_users_df['Count']))
}

print("6.")
print("Top 5 IP:")
print(top_5_ips_df.to_string(index=False))
print("\nTop 5 Users:")
print(top_5_users_df.to_string(index=False))

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
print("\nResults saved to results.json")

1.
  Month Day      Time   Host Process  \
0   Dec  10  06:55:46  LabSZ    sshd   
1   Dec  10  06:55:46  LabSZ    sshd   
2   Dec  10  06:55:46  LabSZ    sshd   
3   Dec  10  06:55:46  LabSZ    sshd   
4   Dec  10  06:55:46  LabSZ    sshd   

                                                 msg  
0  reverse mapping checking getaddrinfo for ns.ma...  
1         Invalid user webmaster from 173.234.31.186  
2  input_userauth_request: invalid user webmaster...  
3      pam_unix(sshd:auth): check pass; user unknown  
4  pam_unix(sshd:auth): authentication failure; l...  


2.
Liczba wierszy z brakującymi danymi: 0


3.
Process
sshd    2000
Name: count, dtype: int64


4.
             Datetime Process  \
0 2025-12-10 06:55:46    sshd   
1 2025-12-10 06:55:46    sshd   
2 2025-12-10 06:55:46    sshd   
3 2025-12-10 06:55:46    sshd   
4 2025-12-10 06:55:46    sshd   

                                                 msg  
0  reverse mapping checking getaddrinfo for ns.ma...  
1         Invali